In [ ]:
# --- Validation and Analysis Notebook ---
# This notebook is for post-HPO workflows:
# 1. Load a completed Optuna study.
# 2. Select the Top-N best performing hyperparameter configurations.
# 3. Generate a new, clean profile containing only these elite models.
# 4. Run the validation profile to re-verify performance.
# 5. (Future) Analyze or ensemble the results of the validation run.


import os
import sys
import shutil
import warnings
import logging
from pathlib import Path
from typing import Optional
import optuna

# Suppress TensorFlow and addon warnings for a cleaner console
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOG_LEVEL'] = '3'
warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    module='tensorflow_addons'
)
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

import pandas as pd
import tensorflow as tf

# --- Suppress TensorFlow Warnings ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow_addons')
tf.get_logger().setLevel('ERROR')

# --- Display Settings ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

# --- Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Project Imports ---
from forecast_pipeline.config import (
    EXPERIMENT_CONFIG_DIR, 
    DEFAULT_DATASET, 
    MAX_WORKERS, 
    DefaultExperimentParams, 
    EXPERIMENTS_OUTPUT_DIR
)
from forecast_pipeline.jobs import generate_jobs, select_data_sources
from forecast_pipeline.runner import execute_jobs_robust
from forecast_pipeline.metrics import collate_robust_results
from forecast_pipeline.io_utils import configure_logging
from common.config_wells import DATA_SOURCES
from hpo.optuna_utils import generate_profile_from_top_trials_legacy
from hpo.validation_reporter import create_validation_report, style_validation_report
from IPython.display import display, HTML
from forecast_pipeline.config import HPO_STUDIES_DIR

In [ ]:
# ==============================================================================
#                     CONFIGURATION PANEL (EDIT HERE)
# ==============================================================================
STUDY_TO_VALIDATE = "VOLVE_15-9-F-14_Seq2Context"
METRIC_USED_FOR_RANKING = "weighted_score"
N_TOP_MODELS_TO_VALIDATE = 10
MODEL_INDEXES_TO_VALIDATE = [0]  # e.g. [0, 3, 9] or [4]. None = all top-N will be validated.
VALIDATION_FIXED_PARAMS = {"seed": 42}
ENSEMBLE_SIZE = 1

# --- Derived Paths ---
VALIDATION_DIRNAME = f"validation_{STUDY_TO_VALIDATE}_top_{N_TOP_MODELS_TO_VALIDATE}"
STUDY_FOLDER = EXPERIMENT_CONFIG_DIR / "results" / VALIDATION_DIRNAME
VALIDATION_PROFILE_PATH = EXPERIMENT_CONFIG_DIR / f"validation_{STUDY_TO_VALIDATE}_top_{N_TOP_MODELS_TO_VALIDATE}.csv"

# ==============================================================================
#                          Pipeline Functions
# ==============================================================================

def clean_previous_validation_folder(folder_path: Path):
    """Remove existing validation folder, if present."""
    if folder_path.exists() and folder_path.is_dir():
        print(f"Removing old validation folder: {folder_path}")
        shutil.rmtree(folder_path)
    else:
        print(f"Folder not found (may have already been removed): {folder_path}")

def run_robust_pipeline(profile_path: str, ensemble_size: int) -> Optional[str]:
    """Profile-driven, robust job runner. Returns run output dir."""
    logging.info(f"--- Starting pipeline in ROBUST mode for profile: {profile_path} ---")
    sources = select_data_sources(DATA_SOURCES, DEFAULT_DATASET)
    default_params = {**vars(DefaultExperimentParams()), "ensemble_models": ensemble_size}
    jobs = generate_jobs(sources, default_params, profile_path=profile_path)

    # first_job_params = jobs[0][2]
    # import json
    # print("Full parameters for the LEGACY run:")
    # print(json.dumps(first_job_params, indent=2, sort_keys=True)) # sort_keys helps comparison
    if not jobs:
        logging.warning("No jobs generated from profile. Exiting.")
        return None

    run_name = Path(profile_path).stem
    run_output_dir = execute_jobs_robust(jobs, "experiments", run_name, MAX_WORKERS)
    logging.info(f"Run complete. Collating results from: {run_output_dir}")

    leaderboard_df = collate_robust_results(run_output_dir)
    if not leaderboard_df.empty:
        leaderboard_path = Path(run_output_dir) / "leaderboard.csv"
        leaderboard_df.to_csv(leaderboard_path, index=False)
        logging.info(f"Leaderboard saved to: {leaderboard_path}")

        
        print(f"\n--- Top {N_TOP_MODELS_TO_VALIDATE} Results (by val_smape_cum) ---")
        required_cols = ['architecture_profile', 
                 'batch_size', 
                 'data_sample', 
                 'epochs', 
                 'lag_window',
                 'learning_rate', 
                 'physics_strategy',
                 'weighted_score',   # <-- essa pode não existir!
                 'val_smape_cum', 
                 'val_smape_agg']

        available_cols = [col for col in required_cols if col in leaderboard_df.columns]
        display(leaderboard_df[available_cols].sort_values(by="val_smape_cum", ascending=True).head(N_TOP_MODELS_TO_VALIDATE))
        

    return run_output_dir

def generate_validation_profile(
    study_name: str,
    n_top: int,
    output_path: Path,
    fixed_params: dict,
    model_indexes: list = None
) -> Optional[pd.DataFrame]:
    """
    Create a validation profile containing only the top-N unique configs.
    If 'model_indexes' is specified, only the selected indexes (in top-N ranking) will be included.
    """
    print(f"Generating validation profile for the top {n_top} trials from study '{study_name}'...")
    top_trials_df = generate_profile_from_top_trials_legacy(
        study_name=study_name,
        n_top_trials=n_top,
        output_file=output_path,
        fixed_params=fixed_params
    )
    print("\n--- TOP Configurations to be Validated ---")
    display(top_trials_df)
    if model_indexes is not None:
        # Select only the rows corresponding to the given indexes in the sorted top-N DataFrame
        print(f"⚡ Selecting specific model indexes for validation: {model_indexes}")
        top_trials_df = top_trials_df.iloc[model_indexes]
        top_trials_df.to_csv(output_path, index=False)
        print("\n--- TOP Configurations ---")
        display(top_trials_df)

    
    return top_trials_df

def display_validation_report(
    hpo_leaderboard_df: pd.DataFrame, 
    validation_leaderboard_df: pd.DataFrame, 
    hyperparameter_cols: list
):
    """Generate and display styled validation report."""
    try:
        report_df = create_validation_report(
            hpo_leaderboard=hpo_leaderboard_df,
            validation_leaderboard=validation_leaderboard_df,
            hyperparameter_cols=hyperparameter_cols
        )
        styled_report = style_validation_report(report_df)
        display(HTML(styled_report.to_html()))
    except Exception as e:
        print(f"❌ Could not generate validation report: {e}")

# ==============================================================================
#                               Main Routine
# ==============================================================================

def main():
    configure_logging()

    # 1. Clean up previous validation folder
    clean_previous_validation_folder(STUDY_FOLDER)

    print("✅ Setup complete. Ready for validation.")

    # 2. Generate a validation profile with top-N configs
    validation_profile_df = generate_validation_profile(
        STUDY_TO_VALIDATE,
        N_TOP_MODELS_TO_VALIDATE,
        VALIDATION_PROFILE_PATH,
        VALIDATION_FIXED_PARAMS,
        model_indexes=MODEL_INDEXES_TO_VALIDATE
    )
    if validation_profile_df is None:
        return

    # 3. Run the validation profile
    print(f"\n🚀 Starting validation run for profile: {VALIDATION_PROFILE_PATH}")
    validation_run_dir = run_robust_pipeline(profile_path=str(VALIDATION_PROFILE_PATH), ensemble_size=ENSEMBLE_SIZE)
    if not validation_run_dir:
        print("\n❌ Validation run failed.")
        return

    print(f"\n✅ Validation run finished. Results are in: {validation_run_dir}")

    # 4. Load HPO and validation leaderboards
    validation_leaderboard_df = collate_robust_results(validation_run_dir)
    save_dir = EXPERIMENTS_OUTPUT_DIR / f"master_leaderboard_{STUDY_TO_VALIDATE}"
    original_hpo_leaderboard_path = save_dir / "leaderboard.csv"
    hpo_leaderboard_df = pd.read_csv(original_hpo_leaderboard_path)

    # 5. Generate the validation analysis report
    if not validation_leaderboard_df.empty and not hpo_leaderboard_df.empty:
        print(f"\n--- Automated Validation Analysis Report ---")
        study = optuna.load_study(study_name=STUDY_TO_VALIDATE, storage=f"sqlite:///{HPO_STUDIES_DIR / STUDY_TO_VALIDATE}.db")
        # Ask the study itself which parameters it optimized
        hyperparameter_cols = list(study.best_trial.params.keys())
        print(f"Discovered hyperparameters from study: {hyperparameter_cols}")
        display_validation_report(
            hpo_leaderboard_df,
            validation_leaderboard_df,
            hyperparameter_cols
        )
    else:
        print("One or both leaderboards are empty. Cannot generate report.")

# --- Entry Point ---
if __name__ == "__main__":
    main()
